## 🚨 URGENT 🚨 

## RQ6609 - 2026 Geolocation and FTE data for SRS Policy

Last updated by AF -> 20th October 2026 

### SDPT Allocation Details

| **Agreed Release Date** | 2nd November 2026 |
|---------|---------|
| **Received** | 22nd September 2026 |
| **Previous RQ** | RQ6204 |
| **Assigned to** | Matthew Gibbins |
| **Reviewer(s)** | Anthony Frost|
| **Completed by Date** | TBC |
| **Time to Complete** | TBC |
| **Review By Date** | TBC |
| **Save Folder** | "O:\Schools Data\6. Data\6.2 Data Requests\RQ6001-7000\RQ6601-6700\RQ6609 2026 Geolocation and FTE data for SRS Policy" |

### Requestor Details

| **Name** | Olivia Heys |
|---------|---------|
| **Email** | Olivia.Heys@education.gov.au |
| **Phone** | (02) 6218 4009 |
| **Job Title (Position)** | Policy Officer |
| **Team Name** | Schooling Resource Standard (SRS) Policy |
| **Organisation Name** | Department of Education |

### Requested Data Items

<br>

| **Filters & Aggregations** | | | | **Data Elements** | | |
|---------|---------|---|---|---------|---------|---|
| **Type** | **Values** |  | # | **Category** | **Name** | **Source** |
| Cohort | All |  | 1 | School | School AGEID | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [AGEID] |
| Sector | All → Gov & Non-gov |  | 2 | School Campus | Location AGEID | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [School_Location_AGEID] |
| Reporting Year | 2026 |  | 3 | School Campus | Main Location Flag | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [IsPrimaryLocation] |
| Student Data | FTE -> funded & FFPOS |  | 4 | Geographic | Latitude & Longitude | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [IGas.Latitude, IGas.Longitude] |
| School Data | Open Schools Only |  | 5 | Geographic | ABS SA1 Code | pr_dap_efs.business_sdpt_nongov_census.`schools_geography_2026` <br> [SA12021.SA1_CODE21] |
| Education Level | All |  | 6 | Student | Total FTE | pr_dap_efs.efs_gold.`ngc_student_cohort_extract` <br> [fte] |
| Aggregation Level | by School Campus |  | 7 | Student | Total FTE | pr_dap_efs.efs_gold.`ngc_student_cohort_extract` <br> [fte] |

### Output Dataset Structure

| # | **Name** | **id** | **Type** | **Description** |
|---|----------|----------|----------|----------|
| 1 | Year | year | int | Calendar year. |
| 2 | School AGEID | school_ageid | int | Australian Government Education Identifier (AGEID) of the school. |
| 3 | Location AGEID | location_ageid | int | Australian Government Education Identifier (AGEID) of the location (campus). |
| 4 | Main Location | main_location | str | Indicates whether the location is the main location of the school. <br> "Yes", "No" |
| 5 | Latitude | latitude | dbl | Geographic latitude of the location measured in decimal degrees north or south of the equator. |
| 6 | Longitude | longitude | dbl | Geographic longitude of the location measured in decimal degrees east or west of the Greenwich Meridian. |
| 7 | ABS SA1 | abs_sa1 | str | Statistical Area 1 code as defined by the ABS Australian Statistical Geography Standard 2021. |
| 8 | FTE  | funded_fte | dbl | The total full time equivalent (FTE) funded students at the school campus, including Indigenous students. Does not include Distance Education and Full Fee Paying Overseas Students. |
| 9 | FFPOS | ffpos_fte | dbl | Reported number of full fee paying overseas students (FTE) |

## Initial Notebook Setup  


In [0]:
# load the required library packages
from pyspark.sql import functions as F
from pyspark.sql.functions import monotonically_increasing_id

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

from pyspark.sql.functions import col, sum, when, count

from functools import reduce

In [0]:
%skip
# parameterise Catalog & Schema to dynamically update when required
dbutils.widgets.text("catalog", "xp_dap_efs")
dbutils.widgets.text("schema1", "business_sdpt_nongov_census")
dbutils.widgets.text("geopgraphy_table", "schools_geography_2026")

dbutils.widgets.text("schema2", "efs_gold")
dbutils.widgets.text("enrolment_table", "ngc_student_cohort")

# parameterise dataframe name to dynamically update when required
dbutils.widgets.text("final_dataset", "schools_df")

## Populate Initial Schools Data

In [0]:
# load the 2026 school data from new geography table filtering for any school which has a closed date
# this should mean we keep open schools only in the list.
schools_df = (
    spark.table("pr_dap_efs.business_sdpt_nongov_census.Schools_Geography_2026")
    .filter(
        F.col("School_location_ClosingDate").isNull()
        & F.col("School_location_EffectiveToDate").isNull()
        & (F.col("School_location_StatusID") == 1)
    )
    .select(
        F.lit(2026).alias("year"),
        F.col("AGEID").cast("int").alias("school_ageid"),
        F.col("School_location_AGEID").cast("int").alias("location_ageid"),
        F.col("IsPrimaryLocation").alias("main_location"),
        F.col("`IGas.Latitude`").alias("latitude"),
        F.col("`IGas.Longitude`").alias("longitude"),
        F.col("`SA12021.SA1_CODE21`").alias("abs_sa1")
    )
)

In [0]:
# 
schools_df = schools_df.withColumn(
    "main_location",
    F.when(F.col("main_location").isNull(), None)
     .when(F.col("main_location"), "Yes")
     .otherwise("No")
)

In [0]:
# create a separate dataframe for duplicate location_ageid's 
duplicate_locations_df = (
    schools_df
    .groupBy("location_ageid")
    .count()
    .filter(F.col("count") > 1)
    .select("location_ageid")
)

schools_duplicates_df = (
    schools_df.alias("s")
    .join(
        duplicate_locations_df.alias("d"),
        on="location_ageid",
        how="inner"
    )
    .orderBy("location_ageid", "school_ageid")
)

# view duplicate dataframe
display(schools_duplicates_df)

In [0]:
# compare all records that share the same location_ageid by grouping the rows for each 
# location and checking which columns contain more than one distinct value.
compare_cols = [
    c for c in schools_duplicates_df.columns
    if c != "location_ageid"
]

for col_name in compare_cols:
    diff_count = (
        schools_duplicates_df
        .groupBy("location_ageid")
        .agg(F.countDistinct(col_name).alias("cnt"))
        .filter(F.col("cnt") > 1)
        .count()
    )

    if diff_count > 0:
        print(f"{col_name}: {diff_count} location_ageid values have differences")
    else:
        schools_df = schools_df.dropDuplicates(["location_ageid"])

# clean up variables no longer required
del col_name, compare_cols, diff_count, duplicate_locations_df, schools_duplicates_df

In [0]:
# count the number of unique school locations (via location_ageid)
print(schools_df.count())

# view dataframe
display(schools_df)

## Find & Populate Additional Schools Data

In [0]:
%skip
# Load ACARA school profile data
acara_df = (
    spark.table("pr_dap_efs.business_sdpt_acara.2025_acara_school_profile_data")
    .select(
        F.col("School AGE ID").cast("string").alias("school_ageid"),
        F.col("ICSEA").cast("int").alias("icsea")
    )
)

# Join onto schools_df
schools_df = (
    schools_df.alias("s")
    .join(
        acara_df.alias("a"),
        on="school_ageid",
        how="left"
    )
)

display(schools_df)

In [0]:
%skip
# Load education level data
education_level_df = (
    spark.table("pr_dap_efs.efs_gold.ngc_student_cohort_extract")
    .select(
        F.col("location_ageid").cast("string").alias("location_ageid"),
        F.col("location_education_level").alias("education_level")
    )
    .filter(F.col("location_ageid").isNotNull())
)

# Check for locations with multiple education levels
education_conflicts_df = (
    education_level_df
    .groupBy("location_ageid")
    .agg(F.countDistinct("education_level").alias("education_level_count"))
    .filter(F.col("education_level_count") > 1)
)

conflict_count = education_conflicts_df.count()

if conflict_count > 0:
    print(f"WARNING: {conflict_count} location_ageid values have multiple education levels")
    display(education_conflicts_df)
else:
    print("No education level conflicts found")

# Keep one education level per location_ageid
education_level_df = (
    education_level_df
    .groupBy("location_ageid")
    .agg(
        F.first("education_level", ignorenulls=True).alias("education_level")
    )
)

# Join to schools_df
schools_df = (
    schools_df
    .join(
        education_level_df,
        on="location_ageid",
        how="left"
    )
)

# delete temporary variables no longer required
del education_level_df, education_conflicts_df, conflict_count

In [0]:
%skip
# place columns & data in order
schools_df = schools_df.select(
    "year",
    "school_ageid",
    "school_name",
    "location_ageid",
    "location_name",
    "main_location",
    "address",
    "suburb",
    "postcode",
    "state_territory",
    "sector",
    "school_type",
    "education_level",
    "icsea",
    "latitude",
    "longitude",
    "electorate"
).orderBy("location_ageid", "school_ageid")

# inspect dataframe
display(schools_df)

# clean up variables no longer required
# del acara_df, address_lookup_df, locations_df
del acara_df

## Populate Non-Governmnt & FTE data

In [0]:
# pr_dap_efs.efs_gold.ngc_student_cohort_extract

student_cohort_df = (
    spark.table("pr_dap_efs.efs_gold.ngc_student_cohort_extract")
    .filter(F.col("CalendarYear") == 26)
    .groupBy("location_ageid")
    .agg(
        # total FTE
        F.sum(
            F.col("fte").cast("double") * F.col("headcount").cast("double")
        ).alias("total_fte"),

        # distance education FTE
        F.sum(
            F.when(
                F.col("education_type") == "Distance Education Student",
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("de_fte"),

        # FFPOS FTE
        F.sum(
            F.when(
                F.col("overseas_student") == "Yes",
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("ffpos_fte"),

        # students who are BOTH DE and FFPOS
        F.sum(
            F.when(
                (F.col("education_type") == "Distance Education Student")
                & (F.col("overseas_student") == "Yes"),
                F.col("fte").cast("double") * F.col("headcount").cast("double")
            )
        ).alias("de_ffpos_fte")
    )
    .fillna({
        "de_fte": 0,
        "ffpos_fte": 0,
        "de_ffpos_fte": 0
    })
    .withColumn(
        "funded_fte",
        F.col("total_fte")
        - F.col("de_fte")
        - F.col("ffpos_fte")
        + F.col("de_ffpos_fte")
    )
    .orderBy("location_ageid")
)

display(student_cohort_df)

In [0]:
# remove rows where these no data (all values are 0)
student_cohort_df = student_cohort_df.filter(
    ~(
        (F.col("total_fte") == 0) &
        (F.col("de_fte") == 0) &
        (F.col("ffpos_fte") == 0) &
        (F.col("de_ffpos_fte") == 0) &
        (F.col("funded_fte") == 0)
    )
)

student_cohort_df = (
    student_cohort_df
    .drop("de_fte", "de_ffpos_fte", "total_fte")
    .select(
        "location_ageid",
        "funded_fte",
        "ffpos_fte"
    )
)

display(student_cohort_df)

## Populate Governmnt FTE data 

In [0]:
# load the government enrolment data file
gov_enrolment_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(
             "file:/Workspace/Users/anthony.frost2@education.gov.au/Data Requests/RQ6619/2026 - All Schools/GSDS_enrolment_output_ALL_prelim.csv"
             
         )
         .select(
             F.col("Location_AGEID").alias("location_ageid"),
             F.col("FTE").alias("fte"),
             F.col("Headcount").alias("headcount"),
             F.col("DE").alias("education_type"),
             F.col("OS_S500(FFPOS)").alias("overseas_student"),
         )
)

display(gov_enrolment_df)

In [0]:
gov_enrolment_df = (
    gov_enrolment_df
    .groupBy("location_ageid")
    .agg(
        # total FTE
        F.sum(
            F.col("fte").cast("double")
        ).alias("total_fte"),

        # distance education FTE
        F.sum(
            F.when(
                F.col("education_type") == "Yes",
                F.col("fte").cast("double")
            )
        ).alias("de_fte"),

        # FFPOS FTE
        F.sum(
            F.when(
                F.col("overseas_student") == "Yes",
                F.col("fte").cast("double")
            )
        ).alias("ffpos_fte"),

        # students who are BOTH DE and FFPOS
        F.sum(
            F.when(
                (F.col("education_type") == "Yes")
                & (F.col("overseas_student") == "Yes"),
                F.col("fte").cast("double")
            )
        ).alias("de_ffpos_fte")
    )
    .fillna({
        "de_fte": 0,
        "ffpos_fte": 0,
        "de_ffpos_fte": 0
    })
    .withColumn(
        "funded_fte",
        F.col("total_fte")
        - F.col("de_fte")
        - F.col("ffpos_fte")
        + F.col("de_ffpos_fte")
    )
    .filter(
        ~(
            (F.col("total_fte") == 0) &
            (F.col("de_fte") == 0) &
            (F.col("ffpos_fte") == 0) &
            (F.col("de_ffpos_fte") == 0) &
            (F.col("funded_fte") == 0)
        )
    )
    .drop("de_fte", "de_ffpos_fte", "total_fte")
    .select(
        "location_ageid",
        "funded_fte",
        "ffpos_fte"
    )
    .orderBy("location_ageid")
)

display(gov_enrolment_df)

In [0]:
# Append the two dataframes
combined_df = (
    student_cohort_df
    .unionByName(
        gov_enrolment_df,
        allowMissingColumns=True
    )
)

display(combined_df)

In [0]:
schools_df = (
    schools_df.alias("s")
    .join(
        combined_df.select(
            "location_ageid",
            "funded_fte",
            "ffpos_fte"
        ).alias("c"),
        on="location_ageid",
        how="full"
    )
    .orderBy("location_ageid")
)

# delete temporary variables no longer required
del student_cohort_df, gov_enrolment_df, combined_df

In [0]:
# change any enrolment of fte values of 0 to null for quick filtering of rows 
schools_df = schools_df.select(
    *[
        F.when(F.col(c) == 0, None).otherwise(F.col(c)).alias(c)
        if c in [
            "funded_fte",
            "ffpos_fte"
        ]
        else F.col(c)
        for c in schools_df.columns
    ]
)

display(schools_df)

# filter out all rows (schools) which have no enrolment data
schools_df = schools_df.dropna(
    how="all",
    subset=[
        "funded_fte",
        "ffpos_fte"
    ]
)

In [0]:
# place columns & data in order
schools_df = schools_df.select(
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "latitude",
    "longitude",
    "abs_sa1",
    "funded_fte",
    "ffpos_fte"
).orderBy("location_ageid", "school_ageid")

# inspect dataframe
display(schools_df)

## Fix Data Issues Identified during Validation

missing school campus data manually taken from EFS Prgoam Admin
<br>
missing SA1 values taken from ABS maps


#### location_ageid "50079"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "year",
        F.when(F.col("location_ageid") == 50079, F.lit(2026))
         .otherwise(F.col("year"))
    )
    .withColumn(
        "school_ageid",
        F.when(F.col("location_ageid") == 50079, F.lit(10079))
         .otherwise(F.col("school_ageid"))
    )
    .withColumn(
        "main_location",
        F.when(F.col("location_ageid") == 50079, F.lit("Yes"))
         .otherwise(F.col("main_location"))
    )
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 50079, F.lit(-35.149079))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 50079, F.lit(150.696121))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 50079, F.lit(90103100301))
         .otherwise(F.col("abs_sa1"))
    )
)

#### location_ageid "101583"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "year",
        F.when(F.col("location_ageid") == 101583, F.lit(2026))
         .otherwise(F.col("year"))
    )
    .withColumn(
        "school_ageid",
        F.when(F.col("location_ageid") == 101583, F.lit(17497))
         .otherwise(F.col("school_ageid"))
    )
    .withColumn(
        "main_location",
        F.when(F.col("location_ageid") == 101583, F.lit("Yes"))
         .otherwise(F.col("main_location"))
    )
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 101583, F.lit(-36.36748))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 101583, F.lit(145.415551))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 101583, F.lit(21603141640))
         .otherwise(F.col("abs_sa1"))
    )
)

#### location+ageid "86780"

In [0]:
schools_df = (
    schools_df
    .withColumn(
        "latitude",
        F.when(F.col("location_ageid") == 86780, F.lit(-27.585908))
         .otherwise(F.col("latitude"))
    )
    .withColumn(
        "longitude",
        F.when(F.col("location_ageid") == 86780, F.lit(153.293470))
         .otherwise(F.col("longitude"))
    )
    .withColumn(
        "abs_sa1",
        F.when(F.col("location_ageid") == 86780, F.lit(30102101333))
         .otherwise(F.col("abs_sa1"))
    )
)

#### Null Handling

ICSEA values missing for 15 schools/school locations should remain as null (or na/not available)

Enrolments and FTE values should be displayed as 0 rather than null

In [0]:
# update any null values to 0 for enrolments and fte
schools_df = schools_df.fillna(
    {
        "funded_fte": 0,
        "ffpos_fte": 0,
    }
)

#### Review dataframe 

In [0]:
# place columns & data in order
schools_df = schools_df.select(
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "funded_fte",
    "ffpos_fte",
    "latitude",
    "longitude",
    "abs_sa1"
).orderBy("location_ageid", "school_ageid")

# inspect dataframe
display(schools_df)

## Dataset Validation

In [0]:
# use this function to display all the distint values of a column, the row count and total for confirmation
def display_value_counts(column_name):
    df = globals()[dbutils.widgets.get("df_to_validate")]

    df_counts = (
        df.groupBy(column_name)
          .agg(F.count("*").alias("Row_Count"))
    )

    df_total = (
        df.agg(F.count("*").alias("row_count"))
          .withColumn(column_name, F.lit("TOTAL"))
          .select(column_name, "row_count")
    )

    display(
        df_counts
            .unionByName(df_total)
            .orderBy(
                F.when(F.col(column_name) == "TOTAL", 1).otherwise(0),
                F.col(column_name)
            )
    )

In [0]:
# check the total number of columns in the final table 
# based on the data request and final structure we are expecting 18

# ============================================
# validation PASS when # of columns == 21
# ============================================
print(f"Number of columns: {len(globals()[dbutils.widgets.get("df_to_validate")].columns)} \n")

In [0]:
# desired column headings, directly copied from the sample dataset supplied
desired_columns = [
    "year",
    "school_ageid",
    "location_ageid",
    "main_location",
    "latitude",
    "longitude",
    "abs_sa1",
    "funded_fte",
    "ffpos_fte"
]

# columns from the generated dataset
actual_columns = globals()[dbutils.widgets.get("df_to_validate")].columns

# display side-by-side comparison
max_len = max(len(desired_columns), len(actual_columns))

comparison_data = [
    (
        desired_columns[i] if i < len(desired_columns) else None,
        actual_columns[i] if i < len(actual_columns) else None
    )
    for i in range(max_len)
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["Desired Column Headings", f"{dbutils.widgets.get('df_to_validate')} Column Headings"]
)

# ========================================================
# validation PASS when visual check confirms columns match
# ========================================================

# view the dataframe results
display(comparison_df)

# variables no longer required
del actual_columns, max_len, comparison_data, comparison_df, desired_columns

In [0]:
# count nulls in each column
null_counts = globals()[dbutils.widgets.get("df_to_validate")].select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in globals()[dbutils.widgets.get("df_to_validate")].columns
])

# transpose null count for easier viewing
null_counts_long = null_counts.selectExpr(
    f"""
    stack(
        {len(globals()[dbutils.widgets.get("df_to_validate")].columns)},
        {', '.join([f"'{c}', `{c}`" for c in globals()[dbutils.widgets.get("df_to_validate")].columns])}
    ) as (ColumnName, NullCount)
    """
)

# view the dataframe results
display(null_counts_long)

# variables no longer required
del null_counts, null_counts_long

# =============================================
# validation PASS when all NullCount values = 0
# =============================================

In [0]:
# view all uniqe values of "year" and confirm they are accurate
# already confirmed there's no null values
display(
    globals()[dbutils.widgets.get("df_to_validate")]
        .select("year")
        .distinct()
        .orderBy("year")
)

# ================================================
# validation PASS when only returned value is 2026
# ================================================

In [0]:
# The following columns are confirmed as having no null values:
#
# school_ageid

In [0]:
# check for any null location_ageid values
null_rows = globals()[dbutils.widgets.get("df_to_validate")].filter(col("location_ageid").isNull())
null_count = null_rows.count()

# check for duplicate location_ageid values
duplicate_ids = (
    globals()[dbutils.widgets.get("df_to_validate")]
    .groupBy("location_ageid")
    .count()
    .filter(col("count") > 1)
)

duplicate_count = duplicate_ids.count()

print(f"Null location_ageid count: {null_count}")
print(f"Duplicate location_ageid count: {duplicate_count}")

# display offending rows
if null_count > 0:
    print("Rows with NULL location_ageid:")
    display(null_rows)

if duplicate_count > 0:
    print("Rows with duplicate location_ageid values:")
    
    duplicate_rows = (
        globals()[dbutils.widgets.get("df_to_validate")].alias("a")
        .join(
            duplicate_ids.select("location_ageid").alias("b"),
            on="location_ageid",
            how="inner"
        )
        .orderBy("location_ageid")
    )
    
    # view the dataframe results
    display(duplicate_rows)

# overall validation result
if null_count == 0 and duplicate_count == 0:
    print("PASS: location_ageid contains no nulls and all values are unique.")
else:
    print("FAIL: Validation checks failed.")

# variables no longer required
del null_rows, null_count, duplicate_ids, duplicate_count

# ===============================================
# validation PASS when no nulls and no duplicates
# ===============================================

In [0]:
# view all unique values of "school_sector" and confirm they are accurate
# already confirmed there's no null values
display_value_counts("main_location")

# ============================================================
# validation PASS when only returned values are "Yes" or "No" 
# ============================================================

In [0]:
# Approximate Australian geographic bounds
# Latitude:  -44 to -10
# Longitude: 112 to 154

display(
    schools_df.filter(
        (F.col("latitude").isNull()) |
        (F.col("longitude").isNull()) |
        (F.col("latitude") < -44) |
        (F.col("latitude") > -10) |
        (F.col("longitude") < 112) |
        (F.col("longitude") > 154)
    )
)

# ============================================================
# validation PASS when no rows are returned for viewing 
# ============================================================


# Display duplicate latitude/longitude pairs and how many records share them
duplicate_coords = (
    schools_df
        .groupBy("latitude", "longitude")
        .count()
        .filter(F.col("count") > 1)
        .select("latitude", "longitude")
)

display(
    schools_df
        .join(
            duplicate_coords,
            on=["latitude", "longitude"],
            how="inner"
        )
        .orderBy("latitude", "longitude", "location_ageid")
)

# delete temporary variables no longer required
del duplicate_coords

# ============================================================
# validation PASS after review of any duplicate pairs
# ============================================================

In [0]:
# these are the columns to summarise
metrics = [
    "funded_fte",
    "ffpos_fte"
]

summary_rows = []

for metric in metrics:
    stats = (
        schools_df.select(
            F.min(metric).alias("Minimum"),
            F.max(metric).alias("Maximum"),
            F.expr(f"percentile_approx({metric}, 0.5)").alias("Median"),
            F.avg(metric).alias("Average"),
            F.sum(metric).alias("Total")
        )
        .collect()[0]
    )

    summary_rows.append(
        (
            metric,
            round(float(stats["Minimum"]), 4) if stats["Minimum"] is not None else None,
            round(float(stats["Maximum"]), 4) if stats["Maximum"] is not None else None,
            round(float(stats["Median"]), 4) if stats["Median"] is not None else None,
            round(float(stats["Average"]), 4) if stats["Average"] is not None else None,
            round(float(stats["Total"]), 4) if stats["Total"] is not None else None,
        )
    )

summary_df = spark.createDataFrame(
    summary_rows,
    [
        "Metric",
        "Minimum",
        "Maximum",
        "Median",
        "Average",
        "Total"
    ]
)

display(summary_df)

# delete temporary variables no longer required
del metric, metrics, summary_rows, summary_df, stats 

# ===============================================================
# validation PASS after a manual sanity check of the below values 
# ===============================================================

## Complete metadata and finalise

In [0]:
display(schools_df)


#### RQ6609 Metadata Table

| Data Item Name | Data Item Type | Data Item Length | Data Item Description / Meaning / Comments / Business Rules | Data Values |
|---|---|---|---|---|
| **School AGEID** | Integer | 5 | Australian Government Education ID (AGEID) of school. | Discrete |
| **Location AGEID** | Integer | 5 | Australian Government Education ID (AGEID) of school location. | Unique, Discrete |
| **Main Location** | Character | 3 | Indicates the main location of the school. | Yes, No |
| **Funded Student FTE** | Decimal | (5,1) | Full-Time Equivalent (FTE) of funded student enrolments. | Continuous |
| **FFPOS FTE** | Decimal | (5,1) | Full-Time Equivalent (FTE) of Full Fee Paying Overseas Students (FFPOS). | Continuous |
| **Latitude** | Decimal | (9,6) | Latitude of the school location as reported on EFS. | Continuous |
| **Longitude** | Decimal | (9,6) | Longitude of the school location as reported on EFS. | Continuous |
| **Statistical Area 2021** | Integer | 11 | Australian Bureau of Statistics (ABS) Australian Statistical Geography Standard (ASGS) Statistical Area Level 1 (SA1) 2021 code. An 11-digit identifier representing the statistical area in which the school is located. | Discrete |

### RQ6609 Notes

1. Latitude and Longitude of the school location are as reported in EFS.
2. ABS Statistical Area 2021 is obtained from the Boundary Selection in the Spatial Enablement Add-in using the latitude/longitude as reported in EFS.
3. Where the latitude and longitude is missing the SA1 2021 is obtained from the location address passed through the Spatial Add-in.
4. Funded Student FTE is the total student FTE excluding FFPOS FTE and Distance Education students
5. FFPOS FTE is the FTE of Full Fee Paying Oversas Students identified where the census response for overseas_student is 'Yes'. These are excluded from the student FTE in the Funded FTE total.
